# Evaluate v4 on the locked 50

Runs the v4 pathology detector (DENTEX + OralXrays + converted Romanian) with
the unchanged tooth detector through the same `evaluate.py` harness, then
compares against v1/v2/v3.

Weights are staged in `/tmp/opencode/v4_weights` and selected with
`DENTEX_MODEL_DIR`, so nothing in `production/` is touched until v4 wins.


In [ ]:
import json
import os
import shutil
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROD = REPO / "dentex_week3" / "production"
DATA = Path("/home/sayfeldin/Projects/dental-model-research")

SRC = Path("/tmp/opencode/kaggle_out/pathology_v4/pathology_v4_best.pt")
assert SRC.exists(), f"run notebooks/fetch_v4.sh first: {SRC}"

V4 = Path("/tmp/opencode/v4_weights")
V4.mkdir(parents=True, exist_ok=True)
shutil.copy2(PROD / "tooth_best.pt", V4 / "tooth_best.pt")
shutil.copy2(SRC, V4 / "disease_best.pt")
os.environ["DENTEX_MODEL_DIR"] = str(V4)

sys.path.insert(0, str(REPO / "dentex_week3"))   # evaluate.py
sys.path.insert(0, str(PROD))                     # inference_pipeline.py
for p in (V4 / "tooth_best.pt", V4 / "disease_best.pt"):
    print(f"{p.name:20s} {p.stat().st_size / 1e6:.1f} MB  {p}")

In [ ]:
import importlib
import inference_pipeline as pipe

importlib.reload(pipe)
assert Path(pipe.MODEL_DIR) == V4, pipe.MODEL_DIR
print("weights dir  :", pipe.MODEL_DIR)
print("tooth model  :", pipe.CONFIG["tooth_model"], "imgsz", pipe.TOOTH_IMGSZ, "conf", pipe.TOOTH_CONF)
print("disease model:", pipe.CONFIG["disease_model"], "imgsz", pipe.DISEASE_IMGSZ, "conf", pipe.DISEASE_CONF)

In [ ]:
VAL_IMAGES = DATA / "data/raw/dentex/validation_data/quadrant_enumeration_disease/xrays"

for name in ("val_0.png", "val_1.png", "val_2.png"):
    out = pipe.run_dentex_pipeline(str(VAL_IMAGES / name))
    print(f"\n{name}: {out['num_detections']} lesions, {len(out['quadrants'])} quadrants")
    for d in out["detections"][:6]:
        print(f"  FDI {d['tooth_fdi']:>2s} {d['disease']:20s} "
              f"p={d['disease_confidence']:.2f} tooth_p={d['tooth_confidence']:.2f}")

In [ ]:
from evaluate import load_ground_truth

GT = DATA / "data/downloads/validation_triple.json"
gt = load_ground_truth(GT)

cache = {}
for i, name in enumerate(sorted(gt), 1):
    try:
        cache[name] = {"image": name, "ok": True,
                       "result": pipe.run_dentex_pipeline(str(VAL_IMAGES / name))}
    except Exception as exc:
        cache[name] = {"image": name, "ok": False, "error": f"{type(exc).__name__}: {exc}"}
    if i % 10 == 0:
        print(f"  {i}/{len(gt)}")

failed = [n for n, r in cache.items() if not r["ok"]]
print(f"scored {len(cache) - len(failed)}/{len(gt)} | failed: {failed}")

In [ ]:
from evaluate import evaluate, render_report

results = evaluate(cache, gt)
report = render_report(results)

out_dir = REPO / "dentex_week3" / "eval"
(out_dir / "evaluation_results_v4.json").write_text(json.dumps(results, indent=2))
(out_dir / "evaluation_report_v4.md").write_text(report)
with open(out_dir / "raw_predictions_v4.jsonl", "w") as fh:
    for name, rec in cache.items():
        fh.write(json.dumps(rec) + "\n")
print(report)

In [ ]:
# ---- v1 vs v2 vs v3 vs v4 ----
v1 = json.loads((REPO / "dentex_week3/eval/evaluation_results.json").read_text())
v2 = json.loads((REPO / "dentex_week3/eval/evaluation_results_v2.json").read_text())
v3 = json.loads((REPO / "dentex_week3/eval/evaluation_results_v3.json").read_text())


def row(label, a, b, c, d):
    print(f"{label:22s} {a:>8.1%} {b:>8.1%} {c:>8.1%} {d:>8.1%} {d - c:>+8.1%}")


print(f"{'metric':22s} {'v1':>8s} {'v2':>8s} {'v3':>8s} {'v4':>8s} {'v4-v3':>8s}")
row("mAP50", v1["mAP50"], v2["mAP50"], v3["mAP50"], results["mAP50"])
row("precision", v1["detection_micro"]["precision"], v2["detection_micro"]["precision"],
    v3["detection_micro"]["precision"], results["detection_micro"]["precision"])
row("recall", v1["detection_micro"]["recall"], v2["detection_micro"]["recall"],
    v3["detection_micro"]["recall"], results["detection_micro"]["recall"])
row("disease acc", v1["label_accuracy"]["disease"]["rate"], v2["label_accuracy"]["disease"]["rate"],
    v3["label_accuracy"]["disease"]["rate"], results["label_accuracy"]["disease"]["rate"])
row("tooth acc", v1["label_accuracy"]["tooth"]["rate"], v2["label_accuracy"]["tooth"]["rate"],
    v3["label_accuracy"]["tooth"]["rate"], results["label_accuracy"]["tooth"]["rate"])
row("joint acc", v1["label_accuracy"]["joint"]["rate"], v2["label_accuracy"]["joint"]["rate"],
    v3["label_accuracy"]["joint"]["rate"], results["label_accuracy"]["joint"]["rate"])

print("\nper-class AP50")
print(f"{'class':22s} {'v1':>8s} {'v2':>8s} {'v3':>8s} {'v4':>8s} {'v4-v3':>8s}")
for cls in results["ap_per_class"]:
    print(f"{cls:22s} {v1['ap_per_class'][cls]['AP50']:>8.1%} "
          f"{v2['ap_per_class'][cls]['AP50']:>8.1%} "
          f"{v3['ap_per_class'][cls]['AP50']:>8.1%} "
          f"{results['ap_per_class'][cls]['AP50']:>8.1%} "
          f"{results['ap_per_class'][cls]['AP50'] - v3['ap_per_class'][cls]['AP50']:>+8.1%}")


In [ ]:
# ---- paired significance: v3 vs v4 ----
import subprocess
cmd = [sys.executable, str(REPO / "dentex_week3/paired_significance.py"),
       "--v1", str(REPO / "dentex_week3/eval/raw_predictions_v3.jsonl"),
       "--v2", str(REPO / "dentex_week3/eval/raw_predictions_v4.jsonl"),
       "--markdown", "--label1", "v3", "--label2", "v4"]
md = subprocess.run(cmd, capture_output=True, text=True, check=True).stdout
(out_dir / "paired_significance_v3_v4.md").write_text(md)
print(md)
